In [98]:
# Imports section
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

## Part 1. Loading the dataset

In [99]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
data = pd.read_csv("science_data_large.csv")
print(data.head(15))

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04


## Part 2. Splitting the dataset

In [100]:
# Take the pandas dataset and split it into our features (X) and label (y)
x = data[["Temperature °C", "Mols KCL"]]
y = data["Size nm^3"]
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

print(y_train)

716    103064.31430
351    193753.02860
936     21670.02857
256    505027.40000
635     75092.02857
           ...     
106     40433.71429
270    335545.00000
860    600674.85710
435    441286.60000
102    505469.25710
Name: Size nm^3, Length: 900, dtype: float64


## Part 3. Perform a Linear Regression

In [101]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(x_train, y_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample = pd.DataFrame({
    "Temperature °C": [500], 
    "Mols KCL": [500]
})
prediction = model.predict(sample)
print("Prediction: ", prediction[0])
# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = model.score(x_test, y_test)
print("Score: ", score)
# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_
intercept = model.intercept_
print(f"Equation: h(x) = {coefficients[0]} * x_1 {'+ ' + str(coefficients[1]) if coefficients[1] >= 0 else '- ' + str(abs(coefficients[1]))} * x_2 {'+ ' + str(intercept) if intercept >= 0 else '- ' + str(abs(intercept))}")

Prediction:  540029.260345451
Score:  0.8552472077276095
Equation: h(x) = 866.1464133719209 * x_1 + 1032.6950664857964 * x_2 - 409391.47958340764


Coefficient of Determination Score: 0.8552472077276095

Coefficient of Determination Meaning: The score measures the how much the independent variable scan explain the variability of the dependent variables. It represents how well the model trained on the training set can predict the labels of the test set. Values vary from 0 to 1, with 1 representing a strong correlation and 0 representing a weak correlation. A score of 0.86 means that approximately 86% of the variance in the slime sizes can be explained by the temperature and KCl concentration. 

Equation: $h(x) = 866.1464133719209 * x_1 + 1032.6950664857964 * x_2 - 409391.47958340764$

## Part 4. Use Cross Validation

In [102]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
cross_validation_scores = cross_val_score(model, x, y, cv=10)
mean_cross_validation_score = np.mean(cross_validation_scores)
std_cross_validation_score = np.std(cross_validation_scores)

# Report on their finding and their significance
print("Cross Validation Scores: ", cross_validation_scores)
print("Mean Cross Validation Score: ", mean_cross_validation_score)
print("Standard Deviation Cross Validation Score: ", std_cross_validation_score)

Cross Validation Scores:  [0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397
 0.87941022 0.86349411 0.78353682 0.88686516]
Mean Cross Validation Score:  0.8552450341984701
Standard Deviation Cross Validation Score:  0.031528762965342454


Cross Validation Scores: 0.81123596, 0.86440978, 0.87808742, 0.86561069, 0.87495621, 0.84484397, 0.87941022, 0.86349411, 0.78353682, 0.88686516

Mean Cross Validation Score: 0.8552450341984701

Standard Deviation Cross Validation Score:  0.031528762965342454

Meaning: The score itself measures how well the model can predict the labels of unseen data. The cross validation score is derived from splitting the entire dataset into $k=10$ subsets, testing the model, trained on every other subset, on each subset. The mean cross validation score is equal to the coefficient of determination, and the standard deviation of the cross validation scores is 0.03, which is means that the scores differs on average from the mean by about 0.03. This signifies that the model is able to represent certain patterns in the data relatively strongly. 

## Part 5. Using Polynomial Regression

In [103]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
polynomial_features = PolynomialFeatures(degree=2)
x_poly_train = polynomial_features.fit_transform(x_train)

polynomial_model = LinearRegression()
polynomial_model.fit(x_poly_train, y_train)

pipeline = make_pipeline(polynomial_features, polynomial_model)

# Report on the metrics and output the resultant equation as you did in Part 3.
x_poly_sample = polynomial_features.transform(sample)
polynomial_model_prediction = polynomial_model.predict(x_poly_sample)
print("Prediction", polynomial_model_prediction[0])

x_poly_test = polynomial_features.transform(x_test)
polynomial_score = polynomial_model.score(x_poly_test, y_test)
print("Score: ", polynomial_score)

polynomial_coefficients = polynomial_model.coef_
polynomial_intercept = polynomial_model.intercept_

def term(coefficient):
    return '+ ' + str(coefficient) if coefficient >= 0 else '- ' + str(abs(coefficient))


print(f"Equation: h(x) = {polynomial_coefficients[3]} * x_1^2 {term(5)} * x_2^2 {term(4)} * x_1x_2 {term(1)} * x_1 {term(2)} * x_2 {term(polynomial_intercept)}")

poly_cross_validation_scores = cross_val_score(pipeline, x, y, cv=10)
mean_poly_cross_validation_score = np.mean(poly_cross_validation_scores)
std_poly_cross_validation_score = np.std(poly_cross_validation_scores)

print("Cross Validation Scores: ", poly_cross_validation_scores)
print("Mean Cross Validation Score: ", mean_poly_cross_validation_score)
print("Standard Deviation Cross Validation Score: ", std_poly_cross_validation_score)

Prediction 513142.8571303543
Score:  1.0
Equation: h(x) = 1.2645884339690383e-11 * x_1^2 + 5 * x_2^2 + 4 * x_1x_2 + 1 * x_1 + 2 * x_2 + 2.0479375962167978e-05
Cross Validation Scores:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Mean Cross Validation Score:  1.0
Standard Deviation Cross Validation Score:  0.0


Coefficient of Determination Score: 1.0

Coefficient of Determination Meaning: A score of 1.0 means that approximately 100% of the variance in the slime sizes can be explained by the temperature and KCl concentration. The model perfectly predicts the size of the slime given the temperature and KCl concentration.

Equation: $h(x) = 1.2645884339690383e-11 * x_1^2 + 5 * x_2^2 + 4 * x_1x_2 + 1 * x_1 + 2 * x_2 + 2.0479375962167978e-05$

Cross Validation Scores: 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

Mean Cross Validation Score: 1

Standard Deviation Cross Validation Score: 0

Cross Validation Scores Meaning: The mean cross validation scores have a mean of 1.0 with a standard deviation of 0.0, which means that the model found an equation that perfectly fits the data. In other words, the model found an exact correlation between the features and the labels and is able to accurately predict labels approximately 100% of the time.